In [ ]:
import torch
import torch.nn as nn

# Download and import the MIT Introduction to Deep Learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class OurDenseLayer(torch.nn.Module):
  def __init__(self, num_inputs, num_outputs):
    super(OurDenseLayer, self).__init__()
    self.W = torch.nn.Parameter(torch.randn(num_inputs,num_outputs))
    self.bias = torch.nn.Parameter(torch.randn(num_outputs))

  def forward(self, x):
    z = torch.matmul(x,self.W) + self.bias
    y = torch.sigmoid(z)
    return y


In [ ]:
num_inputs = 2
num_outputs = 3
layer = OurDenseLayer(num_inputs, num_outputs)
x_input = torch.tensor([[1,2.]])
y = layer(x_input)

print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

In [ ]:
num_inputs = 2
num_outputs = 3

model = nn.Sequential(
    nn.Linear(num_inputs, num_outputs),
    nn.Sigmoid()
)


In [ ]:
x_input = torch.tensor([[1, 2.]])
model_output = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

In [ ]:
##NN
class LinearWithSigmoidActivation(nn.Module):
  def __init__(self, num_inputs, num_outputs):
    super(LinearWithSigmoidActivation, self).__init__()
    self.linear = nn.Linear(num_inputs, num_outputs)
    self.activation = nn.Sigmoid()

  def forward(self,inputs):
    linear_output = self.linear(inputs)
    output = self.activation(linear_output)
    return output




In [ ]:
n_input_nodes = 2
n_output_nodes = 3
model = LinearWithSigmoidActivation(n_input_nodes, n_output_nodes)
x_input = torch.tensor([[1, 2.]])
y = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

In [ ]:
class LinearButSometimesIdentity(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(LinearButSometimesIdentity, self).__init__()
        self.linear = nn.Linear(num_inputs, num_outputs)

    '''TODO: Implement the behavior where the network outputs the input, unchanged,
        under control of the isidentity argument.'''
    def forward(self, inputs, isidentity=False):
        linear_output = self.linear(inputs)
        if isidentity:
          output = inputs
        else:
          output = model(inputs)

        return output


In [ ]:
# Test the IdentityModel
model = LinearButSometimesIdentity(num_inputs=2, num_outputs=3)
x_input = torch.tensor([[1, 2.]])

'''TODO: pass the input into the model and call with and without the input identity option.'''
out_with_linear = LinearWithSigmoidActivation(num_inputs=2, num_outputs=3)(x_input)
out_with_identity = LinearButSometimesIdentity(num_inputs=2, num_outputs=3)(x_input, isidentity=True)

print(f"input: {x_input}")
print("Network linear output: {}; network identity output: {}".format(out_with_linear, out_with_identity))

In [ ]:
#gradien descent
x = torch.tensor(3.0,requires_grad=True)
y = x ** 2
y.backward()

dy_dx = x.grad
print("dy_dx of y=x^2 at x=3.0 is: ", dy_dx)
assert dy_dx == 6.0

In [ ]:
#loss
x = torch.randn(1)
print(f"Initializing x={x.item()}")

learning_rate = 1e-2
history = []
x_f = 4

for i in range(500):
  x = torch.tensor([x],requires_grad=True)

  loss = torch.square(x - x_f)
  loss.backward()

  x = x.item() - learning_rate * x.grad
  history.append(x.item())

plt.plot(history)
plt.plot([0, 500], [x_f, x_f])
plt.legend(('Predicted', 'True'))
plt.xlabel('Iteration')
plt.ylabel('x value')
plt.show()

In [ ]:
#RNN for music generation

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml

# Retrieve API key securely from Colab Secrets (Key icon in the left sidebar)
from google.colab import userdata
api_key = userdata.get('COMET_API_KEY')

# Check that API key was set in Colab Secrets
assert api_key is not None, "Please set COMET_API_KEY in Colab Secrets (Key icon on the left)"

# Initialize Comet
comet_ml.init(api_key=api_key)

# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim

# Download and import the MIT Introduction to Deep Learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1

# Check GPU availability
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)
songs_joined

In [ ]:
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

In [ ]:
#processing
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

In [ ]:
print('{')
for char, _ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
#mapping
def vectorize_string(string):
  result = []
  for char in string:
    result.append(char2idx[char])
  final_result = np.array(result)
  return final_result


vectorized_songs = vectorize_string(songs_joined)
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"





In [ ]:
#training loop
def get_batch(vectorized_songs,seq_length,  batch_size):
  n = vectorized_songs.shape[0] - 1
  idx = np.random.choice(n - seq_length, batch_size)

  input_batch = [vectorized_songs[i : i+seq_length] for i in idx]
  output_batch = [vectorized_songs[i+1 : i+seq_length+1] for i in idx]

  x_batch = torch.tensor(input_batch,dtype=torch.long)
  y_batch = torch.tensor(output_batch,dtype=torch.long)

  return x_batch, y_batch
test_args = (vectorized_songs,10,2)
x_batch , y_batch = get_batch(*test_args)
assert x_batch.shape == (2, 10) , "x batch is incorrect"
assert y_batch.shape == (2, 10) , "y batch is incorrect"
print("batch func works")
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(x_batch[0], y_batch[0])):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx.item()])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx.item()])))

In [ ]:
class LSTMModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_size):
    super(LSTMModel, self).__init__()
    self.hidden_size = hidden_size
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def init_hidden(self, batch_size, device):
    return (torch.zeros(1,batch_size, self.hidden_size).to(device),
            torch.zeros(1,batch_size, self.hidden_size).to(device))

  def forward(self,x,state=None, return_state=False):
    x = self.embedding(x)
    if state is None:
      state = self.init_hidden(x.size(0), x.device)
    out, state = self.lstm(x, state)
    out = self.fc(out)
    return out if not return_state else (out, state)

  # Instantiate the model! Build a simple model with default hyperparameters. You
#     will get the chance to change these later.
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

# print out a summary of the model
print(model)

In [ ]:
# Instantiate the model! Build a simple model with default hyperparameters. You
#     will get the chance to change these later.
vocab_size = len(vocab)
embedding_dim = 256
hidden_size = 1024
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

# print out a summary of the model
print(model)

In [ ]:
# Test the model with some sample data
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
x = x.to(device)
y = y.to(device)

pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

In [ ]:
sampled_indices = torch.multinomial(torch.softmax(pred[0], dim=-1), num_samples=1)
sampled_indices = sampled_indices.squeeze(-1).cpu().numpy()
sampled_indices

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0].cpu()])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

In [ ]:
#train the model
cross_entropy = nn.CrossEntropyLoss()
def compute_loss(labels,logits):
  batched_labels = labels.view(-1)
  batched_logits = logits.view(-1, logits.shape[-1])

  loss = cross_entropy(batched_logits,batched_labels)
  return loss

In [ ]:
y.shape
pred.shape
example_batch_loss = compute_loss(y,pred)
print(f"Prediction shape: {pred.shape} # (batch_size, sequence_length, vocab_size)")
print(f"scalar_loss:      {example_batch_loss.mean().item()}")

In [ ]:
### Hyperparameter setting and optimization ###

vocab_size = len(vocab)

# Model parameters:
params = dict(
  num_training_iterations = 3000,  # Increase this to train longer
  batch_size = 8,  # Experiment between 1 and 64
  seq_length = 100,  # Experiment between 50 and 500
  learning_rate = 5e-3,  # Experiment between 1e-5 and 1e-1
  embedding_dim = 256,
  hidden_size = 1024,  # Experiment between 1 and 2048
)

# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
def create_experiment():
  # end any prior experiments
  if 'experiment' in locals():
    experiment.end()

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(
                  api_key=api_key,
                  project_name="6S191_Lab1_Part2")
  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

In [ ]:
model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])

def train_step(x,y):
  model.train()
  optimizer.zero_grad()

  y_hat = model(x)
  loss = compute_loss(y,y_hat)

  #backward
  loss.backward()
  optimizer.step()

  return loss

In [ ]:
history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for iter in tqdm(range(params["num_training_iterations"])):

    # Grab a batch and propagate it through the network
    x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])

    # Convert numpy arrays to PyTorch tensors
    x_batch = torch.tensor(x_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(y_batch, dtype=torch.long).to(device)

    # Take a train step
    loss = train_step(x_batch, y_batch)

    # Log the loss to the Comet interface
    experiment.log_metric("loss", loss.item(), step=iter)

    # Update the progress bar and visualize within notebook
    history.append(loss.item())
    plotter.plot(history)

    # Save model checkpoint
    if iter % 100 == 0:
        torch.save(model.state_dict(), checkpoint_prefix)

# Save the final trained model
torch.save(model.state_dict(), checkpoint_prefix)
experiment.flush()

In [ ]:
def generate_text(model, start_string, generation_length=1000):
    input_idx = vectorize_string(start_string)
    input_idx = torch.tensor([input_idx],dtype=torch.long).to(device)

    state = model.init_hidden(input_idx.size(0),device)

    text_generated = []
    tqdm._instances.clear()

    for i in tqdm(range(generation_length)):
        predictions , state = model(input_idx,state,return_state=True)

        predictions = predictions.squeeze(0)
        input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)
        text_generated.append(idx2char[input_idx.item()])

    return (start_string + ''.join(text_generated))
generated_text = generate_text(model,start_string="X",generation_length=1000)


In [ ]:
generated_songs = mdl.lab1.extract_song_snippet(generated_text)
for i, song in enumerate(generated_songs):
  waveform = mdl.lab1.play_song(song)
  if waveform:
    print("Genrated song",i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface -- you can access it there
    experiment.log_asset(wav_file_path)

In [ ]:
experiment.end()